In [1]:
import pandas as pd
from pathlib import Path
import calendar

In [2]:
db_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Mensuales")
to_save_path = Path(r"D:\ProyectoAnalisisElectrico\DiaPromedio\Periodos")

In [3]:
def days_in_month(date):
    año = date//100
    mes = date%100
    _, dias = calendar.monthrange(2000+año, mes)
    return dias

In [ ]:
month_number = {
    "enero": "01",
    "febrero": "02",
    "marzo": "03",
    "abril": "04",
    "mayo": "05",
    "junio": "06",
    "julio": "07",
    "agosto": "08",
    "septiembre": "09",
    "octubre": "10",
    "noviembre": "11",
    "diciembre": "12"
}
def get_folders_by_period(desde=None, hasta=None):
    if desde is None and hasta is None:
        return sorted([d.name for d in db_path.iterdir()])
    inicio = f"{desde[1] % 100:02d}{month_number[desde[0]]}"
    fin = f"{hasta[1] % 100:02d}{month_number[hasta[0]]}"
    disponibles = sorted([d.name for d in db_path.iterdir()])
    return disponibles[disponibles.index(inicio) : disponibles.index(fin) + 1]

def build_dynamic_mask(date_column, periods):
    dates_str = date_column.astype(str)
    formatted_dates = dates_str.str[2:4] + dates_str.str[5:7]
    client_dates = set(formatted_dates)
    return "".join("1" if period in client_dates else "0" for period in periods)

In [ ]:
periods = get_folders_by_period()
dfs = []
for i, month in enumerate(periods):
    df_path = db_path/month/f"{month}_mean_month.parquet"
    dfs.append(pd.read_parquet(df_path))
df = pd.concat(dfs)
start = periods[0]
end = periods[-1]
dates = f"{start}_{end}"
df["period"] = dates



In [ ]:
sep = "::"
agg_rules = {
    'medida': ('medida', 'mean'), 
    'CMg[CLP/KWh]': ('CMg[CLP/KWh]', 'mean'),
    'valorizado_CLP': ('valorizado_CLP', 'mean'),
    
    'Calendario_Activo': ('Año_Mes', build_dynamic_mask(periods)),
    
    'RUT': ('RUT', 'last'),
    'rut_log': ('RUT', lambda x: sep.join(x.dropna().astype(str).unique())),
    'n_ruts': ('RUT', 'nunique'), 

    'Razon_Social': ('Razon_Social', 'last'),
    'razon_social_log': ('Razon_Social', lambda x: sep.join(x.dropna().astype(str).unique())),
    'n_razones_sociales': ('Razon_Social', 'nunique'), 
    
    'Nombre_Corto': ('Nombre_Corto', 'last'),
    'nombre_corto_log': ('Nombre_Corto', lambda x: sep.join(x.dropna().astype(str).unique())),
    'n_nombres_cortos': ('Nombre_Corto', 'nunique'), 
    
    'nombre_barra': ('nombre_barra', 'last'),
    'nombre_barra_log': ('nombre_barra', lambda x: sep.join(x.dropna().astype(str).unique())),
    'n_nombres_barra': ('nombre_barra', 'nunique'), 
    
    'tension': ('tension', 'last'),
    'tension_log': ('tension', lambda x: sep.join(x.dropna().astype(str).unique())),
    'n_tensiones': ('tension', 'nunique'), 
    
    'period': ('period', 'last')
}

group_data = [
    'clave',  
    'Zona',   
    'Hora'    
]

In [ ]:
to_save_df = df.groupby(group_data).agg(**agg_rules)
to_save_df.reset_index()

,clave,nombre_barra,tension,Zona,Razon_Social,RUT,Nombre_Corto,Hora,Año_Mes,medida,CMg[CLP/KWh],valorizado_CLP
0,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,0,2025-05-01,-30.637419,75.958843,-2329.032062
1,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,1,2025-05-01,-30.854194,73.731255,-2276.079727
2,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,2,2025-05-01,-31.703226,82.363420,-2607.349204
3,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,3,2025-05-01,-31.179355,83.611068,-2605.479383
4,36142444,A.HOSPICIO,13,Norte Distribución,AES Andes S.A.,94.272.000-9,AES_GENER,4,2025-05-01,-30.407097,76.564499,-2332.254784


In [37]:
to_save_folder = to_save_path/dates
if to_save_folder.is_dir():
        raise RuntimeError(f"Los datos de {dates} ya fueron procesados. Saltando...")
to_save_folder.mkdir(parents=True, exist_ok=True)
to_save_df.to_parquet(to_save_folder / f"{dates}_mean_period.parquet", engine="pyarrow", compression="snappy")